# Session 13: Image Classification and Computer Vision Applications

**Learning goals:**

- Load labelled image files and prepare consistent CNN input tensors.
- Create training, validation and test batches with appropriate transforms and shuffle settings.
- Train and evaluate a ResNet18 classification head for cat and dog images.
- Distinguish image classification, object detection and image generation.
- Explain the basic ideas behind YOLO, DETR, GANs and diffusion models.

**Prerequisites:**

- Basic Python, CNNs and model training.
- Python with PyTorch, torchvision, Matplotlib, scikit-learn and KaggleHub.


## Outline

1. Loading Images with Datasets and Transforms
2. Preparing Image Batches for a CNN Model
3. Image Classification Workflow
4. Case Study: Dog and Cat Classification with ResNet18
5. Object Detection: YOLO and DETR
6. Image Generation: GANs and Diffusion Models

## 1. Loading Images with Datasets and Transforms

ImageFolder pairs each image with its class label. Grayscale, Resize and ToTensor
convert each image into a consistent `[1, 28, 28]` tensor, ready for batching and a CNN.

[Session 13.1: Loading Images with Datasets and Transforms](<Session13.1 Loading Images with Datasets and Transforms.ipynb>).

![Image files are resized and converted to single-channel tensors, then batched for a CNN.](assets/session13-image-to-cnn-pipeline.png)

## 2. Preparing Image Batches for a CNN Model

DataLoader groups image–label pairs into batches, shuffling their order for training.
Here, `batch_size=2` means two images per batch: four images form two batches.
A loop sends one batch at a time to the CNN until all samples have been processed.

[Session 13.2: Preparing Image Batches for a CNN Model](<Session13.2 Preparing Image Batches for a CNN Model.ipynb>).

![Four labelled images are shuffled into two clearly separated batches of two images. Batch 1 contains digits 1 and 0 with matching labels and is passed to the CNN before the next batch.](assets/session13-dataloader-batches.png)


## 3. Image Classification Workflow

Apply fixed image transforms, split the data into training and testing sets,
then batch each set. Train the CNN on training batches, evaluate the trained model
on testing batches using loss, accuracy and F1, then save the trained weights.

![Multiple labelled cat and dog images become tensors, which are split into training and testing sets. Each set is batched; training updates CNN weights, while testing measures loss, accuracy and F1 without weight updates. The trained model is then saved.](assets/session13-image-classification-workflow.png)


## 4. Case Study: Dog and Cat Classification with ResNet18

[Download the ResNet18 cat and dog classification notebook](<Session13.3 ResNet18 Cat and Dog Classification.ipynb>){download="Session13.3 ResNet18 Cat and Dog Classification.ipynb"}

### 4.1 Download the images

Install KaggleHub with `pip install kagglehub` in your terminal if needed. Run this notebook from the AI_SS course folder.
KaggleHub downloads the latest [Dog vs Cat dataset](https://www.kaggle.com/datasets/anthonytherrien/dog-vs-cat) and returns its local folder. The first run needs internet access; later runs reuse the download.

[KaggleHub documentation](https://github.com/Kaggle/kagglehub)

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("anthonytherrien/dog-vs-cat")

print("Path to dataset files:", path)

### 4.2 Prepare training and test batches

Find the folder containing the cat and dog class folders, then select up to 100 images per class for a short classroom run.
Split each class 80/20 for training and testing. Keep these splits fixed when comparing models.

`weights.transforms()` applies the resize, 224 × 224 centre crop and normalisation expected by pretrained ResNet18. These photographs already have three RGB channels.

[ImageFolder documentation](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.ImageFolder.html) · [ResNet18 weights and transforms](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html)

In [ ]:
import torch
from torchvision import datasets
from torchvision.models import ResNet18_Weights
from torch.utils.data import DataLoader, Subset

# Locate the class folders even when the download has a wrapper directory.
class_roots = []
for folder, subfolders, _ in os.walk(path):
    if {name.lower() for name in subfolders} in ({"cat", "dog"}, {"cats", "dogs"}):
        class_roots.append(folder)
if len(class_roots) != 1:
    raise ValueError("Expected one folder containing cat and dog class folders. Inspect: " + path)

SEED = 42
MAX_IMAGES_PER_CLASS = 100  # Limit the classroom example; increase to use more images.
BATCH_SIZE = 16  # Process a small group of images at each training step.
torch.manual_seed(SEED)  # Reproduce the split and the new head's initial weights.
weights = ResNet18_Weights.DEFAULT
image_dataset = datasets.ImageFolder(class_roots[0], transform=weights.transforms())
class_names = image_dataset.classes  # Use ImageFolder's class-to-index order.

train_indices, test_indices = [], []
split_generator = torch.Generator().manual_seed(SEED)
for target in range(len(class_names)):
    indices = [i for i, label in enumerate(image_dataset.targets) if label == target]
    order = torch.randperm(len(indices), generator=split_generator).tolist()
    selected = [indices[i] for i in order[:MAX_IMAGES_PER_CLASS]]
    if len(selected) < 5:
        raise ValueError("Provide at least five images per class.")
    split_at = int(0.8 * len(selected))  # Reserve 20% of each class for testing.
    train_indices.extend(selected[:split_at])
    test_indices.extend(selected[split_at:])

train_loader = DataLoader(
    Subset(image_dataset, train_indices), batch_size=BATCH_SIZE, shuffle=True,
    generator=torch.Generator().manual_seed(SEED),  # Reproduce the shuffled training order.
)
test_dataset = Subset(image_dataset, test_indices)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print("Classes:", class_names)
print("Training images:", len(train_indices), "Test images:", len(test_indices))

### 4.3 Load ResNet18 and configure transfer learning

As in Session 12, replace `fc` with a new classification head, freeze the backbone and give the optimiser only the head's parameters.
The model returns two logits, one for each class. `CrossEntropyLoss` uses these logits directly.
The first run downloads the ImageNet weights.

In [ ]:
from torch import nn
from torchvision.models import resnet18

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

torch.hub.set_dir("../archive/AI_SS/torch")  # Store downloaded pretrained weights locally.
model = resnet18(weights=weights)  # Reuse features learned from ImageNet.
model.fc = nn.Linear(model.fc.in_features, len(class_names))  # Produce cat and dog logits.
model = model.to(device)  # Move parameters before constructing the optimiser.

for parameter in model.parameters():
    parameter.requires_grad = False  # Freeze every parameter first.
for parameter in model.fc.parameters():
    parameter.requires_grad = True  # Train only the replacement head.

LEARNING_RATE = 0.001
loss_function = nn.CrossEntropyLoss()  # Compare logits with integer class labels.
optimiser = torch.optim.Adam(model.fc.parameters(), lr=LEARNING_RATE)
print("Device:", device, "New head:", model.fc)

### 4.4 Train the new classification head

Keep `model.eval()` during head training so the frozen backbone's BatchNorm statistics stay fixed, then select `model.fc.train()` for the head.
Evaluation mode does not disable gradients: `loss.backward()` still calculates them for the trainable head.
Each epoch processes every training batch. The printed loss is the average over all training images.

In [ ]:
EPOCHS = 3  # Make three passes through the training images.
for epoch in range(EPOCHS):
    model.eval()  # Preserve the backbone's BatchNorm running statistics.
    model.fc.train()  # Select training behaviour for the new head.
    total_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimiser.zero_grad()  # Clear gradients from the previous batch.
        logits = model(images)  # Extract features and calculate two class scores.
        loss = loss_function(logits, labels)
        loss.backward()  # Calculate gradients for the head.
        optimiser.step()  # Update only the head's parameters.
        total_loss += loss.item() * labels.size(0)
    print(f"Epoch {epoch + 1}/{EPOCHS}: loss={total_loss / len(train_indices):.3f}")

### 4.5 Evaluate and inspect predictions

Evaluate on the held-out test images. Accuracy reports the proportion of correct predictions; macro F1 gives each class equal weight.
Show five test images with their true and predicted labels. Open the original RGB files for display because the model inputs have been normalised.

Use a separate validation split if you want to tune settings; keep the test set for evaluating the final model.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

model.eval()  # Use fixed layer behaviour for testing.
test_targets, test_predictions = [], []
with torch.no_grad():  # Predict without recording gradients.
    for images, labels in test_loader:
        predictions = model(images.to(device)).argmax(dim=1).cpu()
        test_predictions.extend(predictions.tolist())
        test_targets.extend(labels.tolist())

print(f"Test accuracy: {accuracy_score(test_targets, test_predictions):.1%}")
print(f"Macro F1: {f1_score(test_targets, test_predictions, average='macro', zero_division=0):.3f}")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

DISPLAY_COUNT = min(5, len(test_dataset))
figure, axes = plt.subplots(1, DISPLAY_COUNT, figsize=(12, 3), squeeze=False)
for position, axis in enumerate(axes.flat):
    image_index = test_indices[position]  # Match the fixed test-loader order.
    image_path, actual = image_dataset.samples[image_index]
    with Image.open(image_path) as image:
        axis.imshow(image.convert("RGB"))  # Display the original colour pixels.
    predicted = test_predictions[position]
    axis.set_title(f"True: {class_names[actual]}\nPred: {class_names[predicted]}")
    axis.axis("off")
plt.tight_layout()
plt.show()

## 5. Object Detection: YOLO and DETR

### 5.1 From one label to object locations

Object detection answers two questions: **what objects are present, and where are they?**
Each detected object has a bounding box, a class label and a model score. One image can contain several detections.

![A cat and a dog in one image, each with a bounding box, class label and illustrative model score.](assets/session13-object-detection.png)

The scores are illustrative, rather than measured predictions. To train a detector, we need a class label and bounding box for each object; our cat/dog class folders only supply whole-image labels.

### 5.2 YOLO

YOLO means **You Only Look Once**. A single forward pass extracts image features and predicts candidate boxes and classes at several scales. In the YOLOv8-style pipeline shown below, score filtering and non-maximum suppression (NMS) remove low-scoring and overlapping predictions. Details vary between YOLO versions.

![YOLOv8-style inference: an image passes through CNN features and multi-scale detection heads, then score filtering and NMS produce boxes and class labels.](assets/session13-yolo-principles.png)

**Code example:** [Ultralytics YOLO Python usage](https://docs.ultralytics.com/usage/python/) shows how to load pretrained weights, predict boxes, train and evaluate a detector. Use `yolov8n.pt` to explore the YOLOv8-style model shown in the diagram.

[YOLOv8 model documentation](https://docs.ultralytics.com/models/yolov8/)

### 5.3 DETR

DETR means **DEtection TRansformer**. A CNN extracts image features, a Transformer encoder processes them, and a decoder uses learned **object queries** to predict a set of boxes and classes. Unused prediction slots receive the **no object** class. During training, predictions are matched to labelled objects; original DETR does not need NMS.

![Original DETR inference: CNN features pass through a Transformer encoder and a decoder supplied with learned object queries; prediction slots produce cat, dog or no-object labels, and unused slots are discarded.](assets/session13-detr-principles.png)

**Code examples:** [Official standalone DETR Colab notebook](https://colab.research.google.com/github/facebookresearch/detr/blob/colab/notebooks/detr_demo.ipynb) implements a simplified DETR and visualises predictions. The [official implementation](https://github.com/facebookresearch/detr#notebooks) also links to a hands-on notebook for pretrained inference and attention visualisation.

## 6. Image Generation: GANs and Diffusion Models

### 6.1 Generative adversarial networks

A GAN trains two networks together. The generator maps a latent input to a synthetic image.
The discriminator learns to distinguish real training images from generated ones. Their
competing objectives encourage the generator to produce more realistic samples.

![Early GAN training: the generator produces blurry, unrealistic images and the discriminator may misclassify them; separate losses update each network in alternating steps. After training, the generator produces new images from random latent inputs.](assets/session13-gan-principles.png)

GAN training can be unstable, and a generator may produce a limited variety of outputs
(mode collapse). A basic GAN is not automatically controlled by a text prompt.

**Code examples:** [PyTorch DCGAN tutorial](https://docs.pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html) explains the generator, discriminator and alternating training steps. The [complete PyTorch training script](https://github.com/pytorch/examples/blob/main/dcgan/main.py) provides a runnable implementation. DCGAN is a convolutional GAN.

### 6.2 Diffusion models

During training, noise is added to images at different levels and a neural network learns
a denoising task. During sampling, a learned reverse process starts from noise and gradually
constructs an image. A text-conditioned model also uses a representation of a prompt to guide
generation. Not every diffusion model is text-conditioned.

![Diffusion training adds known noise to images and trains a network to predict it; sampling starts with random noise and applies repeated learned denoising steps to generate a new image.](assets/session13-diffusion-principles.png)

**Code examples:** [Hugging Face Diffusers: unconditional image generation](https://huggingface.co/docs/diffusers/using-diffusers/unconditional_image_generation) provides a short pretrained DDPM example that generates and saves an image without a text prompt. The [complete training script](https://github.com/huggingface/diffusers/blob/main/examples/unconditional_image_generation/train_unconditional.py) shows noise addition, noise prediction and optimisation.
